In [ ]:
# ============================================
# 1. Import Libraries
# ============================================
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap

from sklearn.preprocessing import LabelEncoder, label_binarize
from xgboost import XGBClassifier
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split

# ============================================
# 2. Configure Project Path
# ============================================
PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ============================================
# 3. Import Project Modules
# ============================================
from src.config import (
    TARGET,
    TIER1_FEATURES,
    NUMERIC_FEATURES,
    CATEGORICAL_FEATURES,
)

from src.preprocessing import (
    replace_false_with_nan,
    convert_numeric_columns,
)

from src.models import get_models

from src.benchmark import (
    BalancedSampleWeightClassifier,
    benchmark_models,
    print_benchmark_ranking,
    save_benchmark_results,
)

from src.evaluation import (
    get_per_class_metrics,
    plot_confusion_matrix,
    plot_precision_recall,
    plot_roc,
    plot_calibration,
)

In [ ]:
# ============================================
# 4. Load and Prepare Dataset
# ============================================

df = pd.read_csv(
    PROJECT_ROOT / "data" / "SEED-ML" / "infertility_man_data-v2.csv",
    sep=";",
)

# Extract features and target
X = df[TIER1_FEATURES].copy()
y = df[TARGET].copy()

# Replace encoded missing values
X_clean = replace_false_with_nan(X)

# Convert numeric features to numeric dtype
X_clean = convert_numeric_columns(
    X_clean,
    NUMERIC_FEATURES,
)

In [ ]:
# ============================================
# 5. Define Preprocessing Pipeline
# ============================================

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, NUMERIC_FEATURES),
        ("cat", categorical_transformer, CATEGORICAL_FEATURES),
    ]
)

In [ ]:
# ============================================
# 6. Benchmark Machine Learning Models
# ============================================

# Encode target labels for model training
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# Initialize benchmark models
models = get_models(preprocessor)

print("Models:", list(models.keys()))

# Run repeated stratified cross-validation benchmark
benchmark_results = benchmark_models(
    models=models,
    X=X_clean,
    y=y_encoded,
    n_splits=5,
    n_repeats=5,
    random_state=42,
)

benchmark_results

In [ ]:
# ============================================
# 7. Train Final LightGBM Model
# ============================================

X_train, X_test, y_train, y_test = train_test_split(
    X_clean,
    y_encoded,
    test_size=0.20,
    stratify=y_encoded,
    random_state=42,
)

# Reinitialize models
models = get_models(preprocessor)

# Select the top-performing model
lightgbm_model = models["LightGBM"]

# Fit model on the training set
lightgbm_model.fit(
    X_train,
    y_train,
)

lightgbm_fitted = lightgbm_model

In [ ]:
# ============================================
# 8. Confusion Matrix and Per-Class Performance
# ============================================

class_names = [
    "AT",
    "A",
    "AZ",
    "NO",
    "OAT",
    "OA",
    "OT",
    "O",
    "T",
]

fig, ax = plot_confusion_matrix(
    model=lightgbm_model,
    X_test=X_test,
    y_test=y_test,
    class_names=class_names,
    normalize="true",
)

plt.show()

per_class_metrics = get_per_class_metrics(
    model=lightgbm_model,
    X_test=X_test,
    y_test=y_test,
    class_names=label_encoder.classes_,
)

per_class_metrics

In [ ]:
# ============================================
# 9. Calibration Analysis
# ============================================

fig, ax = plot_calibration(
    model=lightgbm_model,
    X_test=X_test,
    y_test=y_test,
    class_names=class_names,
    n_bins=10,
)

plt.show()

In [ ]:
# ============================================
# 10. Prepare SHAP Explainability Analysis
# ============================================

# Extract fitted preprocessing and classifier steps
preprocessor_fitted = lightgbm_model.named_steps["preprocessor"]
classifier_fitted = lightgbm_model.named_steps["classifier"]

# Transform held-out test features
X_test_transformed = preprocessor_fitted.transform(X_test)

# Retrieve transformed feature names
feature_names = preprocessor_fitted.get_feature_names_out()

X_test_shap = pd.DataFrame(
    X_test_transformed,
    columns=feature_names,
    index=X_test.index,
)

# Compute SHAP values
explainer = shap.TreeExplainer(classifier_fitted)
shap_values = explainer(X_test_shap)

print("SHAP values shape:", shap_values.shape)

In [ ]:
# ============================================
# 11. Global SHAP Feature Importance
# ============================================

# Publication-friendly feature labels
feature_labels = {
    "num__sample_concentration_initial": "Sperm concentration",
    "num__sample_vol_initial": "Semen volume",
    "num__sample_morpho_normal": "Normal morphology",
    "num__sample_num_prog_mob_total": "Progressive motile sperm",
    "num__sample_production_total": "Total sperm count",
    "num__sample_ph": "pH",
    "num__sample_cells_round": "Round cells",
    "num__sample_vitality": "Vitality",
    "num__sample_leukocytes": "Leukocytes",
    "num__sample_survival_test": "Survival test",
    "num__sample_morpho_kruger": "Kruger morphology",

    "cat__sample_red_blood_cells_INCOMPLETE": "Red blood cells (Incomplete)",
    "cat__sample_red_blood_cells_INCOMPLETE ": "Red blood cells (Incomplete)",
    "cat__sample_red_blood_cells_COMPLETE": "Red blood cells (Complete)",

    "cat__sample_liquefaction_COMPLETE": "Complete liquefaction",
    "cat__sample_liquefaction_INCOMPLETE": "Incomplete liquefaction",

    "cat__sample_bodies_gelatinous_0": "Gelatinous bodies",
    "cat__sample_bodies_gelatinous_FALSO": "Gelatinous bodies (No)",
    "cat__sample_bodies_gelatinous_True": "Gelatinous bodies (Yes)",
    "cat__sample_bodies_gelatinous_VERDADERO": "Gelatinous bodies (Yes)",

    "cat__sample_viscosity_NORMAL": "Normal viscosity",
    "cat__sample_viscosity_INCREASED": "Increased viscosity",

    "cat__sample_appearance_NORMAL": "Normal appearance",
    "cat__sample_appearance_WHITE": "White appearance",
    "cat__sample_appearance_TRANSLUCENT": "Translucent appearance",
    "cat__sample_appearance_YELLOWISH": "Yellowish appearance",

    "cat__sample_agglutination_-": "Agglutination (-)",
    "cat__sample_agglutination_+": "Agglutination (+)",
    "cat__sample_agglutination_++": "Agglutination (++)",

    "cat__sample_state_AVERAGE LOSS": "Sample state: Average loss",
    "cat__sample_state_COMPLETE SAMPLE": "Sample state: Complete",
}

# Calculate global mean absolute SHAP values across
# samples and diagnostic classes
mean_abs_shap = np.abs(shap_values.values).mean(axis=(0, 2))

shap_importance = pd.DataFrame({
    "Feature": feature_names,
    "Mean Absolute SHAP": mean_abs_shap,
})

# Apply publication-friendly labels
shap_importance["Feature"] = (
    shap_importance["Feature"]
    .replace(feature_labels)
)

# Rank features by global importance
shap_importance = (
    shap_importance
    .sort_values("Mean Absolute SHAP", ascending=False)
    .reset_index(drop=True)
)

top20 = shap_importance.head(20)

# ============================================
# Plot Top 20 Features
# ============================================

fig, ax = plt.subplots(figsize=(9, 8))

ax.barh(
    top20["Feature"][::-1],
    top20["Mean Absolute SHAP"][::-1],
)

ax.set_xlabel("Mean |SHAP| value", fontsize=12)
ax.set_ylabel("")
ax.set_title(
    "Global SHAP Feature Importance",
    fontsize=15,
    fontweight="bold",
    pad=12,
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(axis="x", alpha=0.2)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================
# 12. SHAP Beeswarm Plot (Azoospermia)
# ============================================

# Select the diagnostic class
class_idx = list(label_encoder.classes_).index("AZOOSPERMIA")

# Convert feature names to publication-friendly labels
pretty_feature_names = [
    feature_labels.get(feature.strip(), feature)
    for feature in X_test_shap.columns
]

# Create SHAP explanation object for the selected class
shap_az = shap.Explanation(
    values=shap_values.values[:, :, class_idx],
    base_values=shap_values.base_values[:, class_idx],
    data=X_test_shap.values,
    feature_names=pretty_feature_names,
)

# Generate beeswarm plot
shap.plots.beeswarm(
    shap_az,
    max_display=50,
)

In [ ]:
# ============================================
# 13. ROC and Precision–Recall Curves
# ============================================

fig, axes = plt.subplots(
    1,
    2,
    figsize=(16, 6),
)

plot_roc(
    model=lightgbm_model,
    X_test=X_test,
    y_test=y_test,
    class_names=class_names,
    ax=axes[0],
)

plot_precision_recall(
    model=lightgbm_model,
    X_test=X_test,
    y_test=y_test,
    class_names=class_names,
    ax=axes[1],
)

plt.tight_layout()
plt.show()

In [18]:
# ============================================
# 14. Multiclass Brier Score
# ============================================

# Predicted class probabilities
y_prob = lightgbm_model.predict_proba(X_test)

# One-hot encode true labels
y_true_onehot = label_binarize(
    y_test,
    classes=np.arange(len(label_encoder.classes_)),
)

# Calculate multiclass Brier score
brier_score = np.mean(
    np.sum((y_prob - y_true_onehot) ** 2, axis=1)
)

print(f"Multiclass Brier Score: {brier_score:.4f}")

Multiclass Brier Score: 0.1380
